# Scan all rules and download each result as soon as it finishes

This is a fixed-CoT residual scan for the four trained rules. Each rule runs independently and is archived immediately after completion, so a later failure does not lose earlier results. Set `LAYERS` to the late-layer range you want. Free generation and combination sweeps are intentionally separate because they multiply runtime.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate pyyaml tqdm matplotlib plotly
from google.colab import files
from pathlib import Path
import hashlib, json, os, sys, zipfile
ROOT=Path('/content/stegano_experiments'); ROOT.mkdir(exist_ok=True)
print('Upload stegano_experiments_bundle.zip')
uploaded=files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as z:
    for name in z.namelist(): assert (ROOT/name).resolve().is_relative_to(ROOT.resolve()), name
    z.extractall(ROOT)
os.chdir(ROOT); sys.path.insert(0,str(ROOT))
for name,digest in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest, name
from experiments.data import config
from experiments.colab import ensure_adapters, download
cfg=config()

In [ ]:
RULES=['s1','voice','clause','lexical']
LAYERS=[20,21,22,23]  # zero-based Qwen2.5-0.5B blocks
METHOD='mean'
RANK=1
OUT_ROOT=Path('incremental_residual_scans')
OUT_ROOT.mkdir(exist_ok=True)
ensure_adapters(cfg,RULES)
print('Rules:', RULES, 'layers:', LAYERS)

In [ ]:
from experiments.residual import scan

for rule in RULES:
    out=OUT_ROOT/f'{rule}_l20plus'
    archive_name=f'{rule}_residual_scan_l20plus'
    if (out/'experiment.json').exists():
        print(f'{rule}: already complete; downloading existing result')
        download(out, archive_name)
        continue
    print(f'\n=== starting {rule} ===', flush=True)
    scan(cfg,rule,LAYERS,out,rank=RANK,method=METHOD)
    print(f'=== finished {rule}; downloading now ===', flush=True)
    download(out, archive_name)
    print(f'=== local copy requested for {rule}; continuing ===', flush=True)

The browser may ask to allow multiple downloads. If the runtime is interrupted, rerun the last cell: completed rules are detected by `experiment.json` and downloaded again rather than recomputed. To use multiple GPUs, run this same notebook in separate Colab sessions with disjoint `RULES` lists; one Colab runtime exposes one GPU to this process.